# 1kg_eur - pinpointing the MAF band where the strange loadings appear

Non-HM3 arm only. Replaces `rare_pca_fractionate.ipynb` and
`rare_pca_plots.ipynb` (both in git history if needed).

**One question:** across 0.1% <= MAF < 1%, where does the loading pathology
start? Eight equal-count MAC bands, one PCA each, then the same three statistics
per band so the onset is a number rather than an impression.

### Why the metadata arm is dropped

`rare_pca_fractionate.ipynb` regressed squared loadings on every per-variant
feature we can compute - call rate, missingness, HWE midp and het excess, indel,
transition, CpG-like, local density, HM3 membership. Result, in the lowest MAF
bin where frequency is nearly constant:

| feature | best partial R^2 | AUC |
|---|---|---|
| `log10_mac` / `maf` | 0.0048 | 0.32-0.58 |
| `is_cpg_like` | 0.0010 | 0.47-0.49 |
| `local_density` | 0.0002 | 0.42-0.47 |
| `obs_ct`, `f_miss` | 0.0003 | 0.49-0.51 |

Nothing. Every AUC is within ~0.08 of 0.5 and no partial R^2 reaches 0.5%. The
genotyping-error and mapping-quality proxies are **not** what is driving the
loadings, so that line is closed and only the frequency axis is carried forward.

### The three statistics

- `enrichment` - share of squared loading mass in the top 1% of variants, divided
  by a **simulated null at that band's variant count**. For iid Gaussian loadings
  the top 1% already carries ~8% of the mass, so the naive "uniform = 1%"
  reference is wrong; equal-count bands also keep the null near-constant across
  bands, which is the point of sizing them that way.
- `max_abs_z` - most extreme individual on PC1-10, in SD.
- `n_beyond_5sd` - individuals past 5 SD on any of PC1-10.

Sensitivity of `enrichment`, measured on synthetic loadings at n = 900: 3
variants at 20x gives 6.9, 5 variants at 5x gives 1.88, unstructured gives ~1.0.
Treat >1.5 as real.

**Runs locally**, off the existing panel and `nonhm3_rare_all.prune.in`. Eight
PCA fits of ~22K variants each.

## Config

In [ ]:
import os, re, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

SAMPLE_SET = "1kg_eur"
WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
OUT_GS = f"{WS_GS}/{SAMPLE_SET}/01_ancestry/rare_pca_bands"

RARE  = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
PANEL = f"{RARE}/qc/rare_qc"
ARM   = "nonhm3_rare_all"
PRUNE_IN = f"{RARE}/{ARM}.prune.in"
KEEP     = f"{RARE}/round2_keep.txt"
FULL_FIT = f"{RARE}/unrel/{ARM}_all"      # reference: the whole arm

OUT = f"{RARE}/bands"
os.makedirs(OUT, exist_ok=True)

N_BANDS     = 8      # equal variant count, so the null stays comparable
N_PCS       = 10
ENRICH_FLAG = 1.5    # above this, call it real

for p in (f"{PANEL}.pgen", PRUNE_IN, KEEP, f"{FULL_FIT}.acount"):
    print(f"  {'ok  ' if os.path.isfile(p) else 'MISS'} {p}")
print(f"\nout: {OUT}")

## Define the bands

MAF comes from the full-arm fit's `.acount`, so the bands partition exactly the
variant set that produced the original loadings. Equal variant counts, not equal
MAF width - a band with fewer variants would have a higher null and look
concentrated for that reason alone.

In [ ]:
a = pd.read_csv(f"{FULL_FIT}.acount", sep=r"\s+")
alt = a["ALT_CTS"] if "ALT_CTS" in a.columns else a["ALT_CT"]
v = pd.DataFrame({"ID": a["ID"],
                  "mac": np.minimum(alt, a["OBS_CT"] - alt).astype(float),
                  "obs_ct": a["OBS_CT"].astype(float)})
v["maf"] = v["mac"] / v["obs_ct"]
v = v.sort_values("maf").reset_index(drop=True)
print(f"arm variants: {len(v):,}   MAF {v['maf'].min():.5f}-{v['maf'].max():.5f}"
      f"   MAC {v['mac'].min():.0f}-{v['mac'].max():.0f}")

bands = {}
for b, idx in enumerate(np.array_split(np.arange(len(v)), N_BANDS), 1):
    sub = v.iloc[idx]
    name = f"band{b}"
    ids = f"{OUT}/{name}.ids"
    sub["ID"].to_csv(ids, index=False, header=False)
    bands[name] = {"band": b, "n": len(sub), "ids": ids,
                   "maf_lo": sub["maf"].min(), "maf_hi": sub["maf"].max(),
                   "mac_lo": sub["mac"].min(), "mac_hi": sub["mac"].max()}
bt = pd.DataFrame(bands).T
print()
print(bt[["band", "n", "maf_lo", "maf_hi", "mac_lo", "mac_hi"]].to_string())

## Fit one PCA per band

In [ ]:
for name, b in bands.items():
    stem = f"{OUT}/{name}"
    if os.path.isfile(f"{stem}.eigenvec"):
        print(f"exists, skipping: {name}")
        continue
    print(f"=== {name}  n={b['n']:,}  MAF {b['maf_lo']:.5f}-{b['maf_hi']:.5f}")
    subprocess.run(["bash", "-c", f"""
set -eo pipefail
export PATH="$HOME/bin:$PATH"
plink2 --pfile "{PANEL}" --nonfounders --keep "{KEEP}" --extract "{b['ids']}" \
  --freq counts --pca approx {N_PCS} allele-wts \
  --threads $(nproc) --out "{stem}"
"""], check=True)

## Load

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS + 1)]

def read_eigenvec(stem):
    d = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    return d.set_index("person_id")[[c for c in PC_COLS if c in d.columns]]

def read_weights(stem):
    L = pd.read_csv(f"{stem}.eigenvec.allele", sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    parts = L[idc].str.split(":", n=2, expand=True)
    L = L.rename(columns={idc: "ID"})
    L["CHROM"] = parts[0].astype(str).str.replace("^chr", "", regex=True)
    L["POS"] = pd.to_numeric(parts[1], errors="coerce")
    L = L[L["CHROM"].str.fullmatch(r"\d+")].dropna(subset=["POS"])
    L["CHROM_N"] = L["CHROM"].astype(int)
    return L.sort_values(["CHROM_N", "POS"]).reset_index(drop=True)

KEYS, E, V, W, LAB = [], {}, {}, {}, {}
for name, b in bands.items():
    stem = f"{OUT}/{name}"
    if not os.path.isfile(f"{stem}.eigenvec"):
        continue
    KEYS.append(name)
    E[name] = read_eigenvec(stem)
    ev = np.atleast_1d(np.loadtxt(f"{stem}.eigenval"))
    V[name] = ev / ev.sum() * 100
    W[name] = read_weights(stem)
    LAB[name] = (f"band {b['band']}\n{b['maf_lo']*100:.3f}-{b['maf_hi']*100:.3f}%\n"
                 f"MAC {b['mac_lo']:.0f}-{b['mac_hi']:.0f}")
    print(f"{name}  n={len(E[name]):>7,}  variants={len(W[name]):>6,}  "
          f"MAF {b['maf_lo']:.5f}-{b['maf_hi']:.5f}")

if os.path.isfile(f"{FULL_FIT}.eigenvec"):
    E["full arm"] = read_eigenvec(FULL_FIT)
    ev = np.atleast_1d(np.loadtxt(f"{FULL_FIT}.eigenval"))
    V["full arm"] = ev / ev.sum() * 100
    W["full arm"] = read_weights(FULL_FIT)
    LAB["full arm"] = f"full arm\n{len(W['full arm']):,} variants"
    print(f"full arm  variants={len(W['full arm']):,}")

## The three statistics, per band

This table is the answer to the question. Read down `enrich_max` for the band
where it first crosses the flag.

In [ ]:
RNG = np.random.default_rng(0)

def top1pct(w2):
    n1 = max(1, len(w2) // 100)
    return np.sort(w2)[-n1:].sum() / w2.sum()

_null_cache = {}
def null_top1pct(nvar, reps=40):
    if nvar not in _null_cache:
        _null_cache[nvar] = float(np.mean(
            [top1pct(RNG.normal(size=nvar) ** 2) for _ in range(reps)]))
    return _null_cache[nvar]

rows = []
for k in KEYS + (["full arm"] if "full arm" in W else []):
    L, d = W[k], E[k]
    null = null_top1pct(len(L))
    enr = {}
    for r in range(1, N_PCS + 1):
        col = f"PC{r}"
        if col in L.columns:
            enr[r] = top1pct(L[col].to_numpy() ** 2) / null
    z = ((d - d.mean()) / d.std()).abs()
    b = bands.get(k, {})
    rows.append({
        "set": k, "band": b.get("band", np.nan), "n_variants": len(L),
        "maf_lo": b.get("maf_lo", np.nan), "maf_hi": b.get("maf_hi", np.nan),
        "mac_lo": b.get("mac_lo", np.nan), "mac_hi": b.get("mac_hi", np.nan),
        "null": null,
        "enrich_pc1": enr.get(1, np.nan),
        "enrich_max": max(enr.values()) if enr else np.nan,
        "enrich_max_pc": max(enr, key=enr.get) if enr else np.nan,
        "max_abs_z": z.to_numpy().max(),
        "n_beyond_5sd": int((z > 5).any(axis=1).sum()),
        "pc1_pct": V[k][0],
    })
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))

In [ ]:
# per-PC enrichment, bands x PCs
rows = []
for k in KEYS + (["full arm"] if "full arm" in W else []):
    L = W[k]; null = null_top1pct(len(L))
    for r in range(1, N_PCS + 1):
        col = f"PC{r}"
        if col in L.columns:
            rows.append({"set": k, "PC": r,
                         "enrichment": top1pct(L[col].to_numpy() ** 2) / null})
per_pc = pd.DataFrame(rows)
piv = per_pc.pivot(index="PC", columns="set", values="enrichment")
order = [k for k in KEYS if k in piv.columns] + (["full arm"] if "full arm" in piv.columns else [])
piv = piv[order]
print("enrichment over null, PC x band:")
print(piv.round(2).to_string())
print(f"\ncells above {ENRICH_FLAG}:")
hits = [(int(r), c, round(piv.loc[r, c], 2))
        for r in piv.index for c in piv.columns if piv.loc[r, c] > ENRICH_FLAG]
print(hits if hits else "  none - no band shows concentrated loadings")

## The pinpoint plot

`enrichment`, `max_abs_z` and `n_beyond_5sd` against band MAF, with the full-arm
fit as a horizontal reference. A step up at one band localises the onset; a flat
line means the effect is not frequency-dependent within this range.

In [ ]:
bb = res[res["band"].notna()].sort_values("maf_lo")
full = res[res["set"] == "full arm"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, col, ttl, ref in zip(
        axes,
        ["enrich_max", "max_abs_z", "n_beyond_5sd"],
        ["loading concentration (x null)", "most extreme individual (SD)",
         "individuals beyond 5 SD"],
        [ENRICH_FLAG, 5, None]):
    mid = np.sqrt(bb["maf_lo"] * bb["maf_hi"])   # geometric midpoint
    ax.plot(mid, bb[col], "-o", ms=5, color="tab:blue")
    for x, y, b in zip(mid, bb[col], bb["band"]):
        ax.annotate(f"b{int(b)}", (x, y), fontsize=6,
                    textcoords="offset points", xytext=(0, 6), ha="center")
    if len(full):
        ax.axhline(full[col].iloc[0], color="0.5", ls=":", lw=1.2, label="full arm")
    if ref is not None:
        ax.axhline(ref, color="crimson", ls="--", lw=1, label="flag")
    ax.set_xscale("log"); ax.set_xlabel("band MAF (geometric midpoint, log)")
    ax.set_title(ttl); ax.legend(fontsize=7)
plt.suptitle(f"{ARM} - where in the frequency range does the phenomenon appear?")
plt.tight_layout(); plt.savefig(f"{OUT}/pinpoint.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.4))
A = piv.to_numpy()
im = ax.imshow(A, aspect="auto", cmap="magma", vmin=min(1.0, np.nanmin(A)))
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels([LAB[c].replace("\n", " ") for c in piv.columns],
                   fontsize=6, rotation=35, ha="right")
ax.set_yticks(range(N_PCS)); ax.set_yticklabels(range(1, N_PCS + 1))
ax.set_ylabel("PC"); ax.set_title("enrichment over null, PC x band")
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        if np.isfinite(A[i, j]):
            ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if A[i, j] < np.nanmax(A) / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout(); plt.savefig(f"{OUT}/enrichment_heatmap.png", dpi=150); plt.show()

## Scree by band

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for k in piv.columns:
    ax.plot(range(1, len(V[k]) + 1), V[k], "-o", ms=4,
            label=LAB[k].replace("\n", " "), lw=1.2)
ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
ax.set_xticks(range(1, N_PCS + 1)); ax.legend(fontsize=6)
ax.set_title(f"{ARM} - eigenvalue spectrum by band")
plt.tight_layout(); plt.savefig(f"{OUT}/scree.png", dpi=150); plt.show()

## Loadings by band, PC1-10

Rows are PCs, columns are bands. |weight| against genomic position, chromosomes
alternating. Each row shares its y-axis across bands so heights are comparable.

In [ ]:
def manhattan(ax, L, col, ymax=None):
    off, ticks, labels = 0.0, [], []
    for c in sorted(L["CHROM_N"].unique()):
        g = L[L["CHROM_N"] == c]
        ax.scatter(off + g["POS"].to_numpy(), g[col].abs(),
                   s=0.35, alpha=0.5, rasterized=True,
                   color="0.25" if c % 2 else "tab:blue")
        ticks.append(off + g["POS"].mean()); labels.append(str(c))
        off += g["POS"].max() + 2e7
    ax.set_xlim(0, off); ax.set_xticks(ticks)
    ax.set_xticklabels(labels, fontsize=5)
    if ymax:
        ax.set_ylim(0, ymax * 1.05)

cols = list(piv.columns)
fig, axes = plt.subplots(N_PCS, len(cols),
                         figsize=(3.6 * len(cols), 1.5 * N_PCS), squeeze=False)
for r in range(N_PCS):
    col = f"PC{r+1}"
    ymax = max(W[k][col].abs().max() for k in cols if col in W[k].columns)
    for c, k in enumerate(cols):
        ax = axes[r, c]
        if col not in W[k].columns:
            ax.axis("off"); continue
        manhattan(ax, W[k], col, ymax)
        if c == 0:
            ax.set_ylabel(col, fontsize=9)
        else:
            ax.set_yticklabels([])
        if r == 0:
            ax.set_title(LAB[k], fontsize=6.5)
        if r < N_PCS - 1:
            ax.set_xticklabels([])
plt.suptitle(f"{ARM} - |allele weight| by position, PC1-{N_PCS}, by MAF band",
             y=0.999)
plt.tight_layout()
plt.savefig(f"{OUT}/loadings_by_band.png", dpi=140, bbox_inches="tight")
plt.show()

## PC scatters by band, PC1-10

Hexbin with log counts - at 223K individuals a scatter hides exactly the isolated
cells that outliers and family clusters produce.

In [ ]:
PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)]
fig, axes = plt.subplots(len(cols), len(PAIRS),
                         figsize=(3.4 * len(PAIRS), 3.0 * len(cols)), squeeze=False)
for r, k in enumerate(cols):
    d = E[k]
    for c, (i, j) in enumerate(PAIRS):
        ax = axes[r, c]
        if f"PC{i}" not in d.columns or f"PC{j}" not in d.columns:
            ax.axis("off"); continue
        hb = ax.hexbin(d[f"PC{i}"], d[f"PC{j}"], gridsize=90, norm=LogNorm(),
                       cmap="viridis", mincnt=1)
        ax.set_xlabel(f"PC{i}", fontsize=8); ax.set_ylabel(f"PC{j}", fontsize=8)
        if c == 0:
            ax.text(-0.34, 0.5, LAB[k], transform=ax.transAxes, fontsize=7,
                    rotation=90, va="center", ha="center")
        if c == len(PAIRS) - 1:
            fig.colorbar(hb, ax=ax, fraction=0.04)
plt.suptitle(f"{ARM} - PC scatters by MAF band", y=0.999)
plt.tight_layout()
plt.savefig(f"{OUT}/pcpairs_by_band.png", dpi=140, bbox_inches="tight")
plt.show()

## Verdict

Mechanical reading of the table, so the conclusion is not eyeballed. Three
possible shapes:

- **onset** - enrichment crosses the flag at one band and stays there. That band
  is the answer.
- **flat above the flag** - present at every frequency, so it is not a
  frequency-dependent artefact.
- **flat below the flag** - no concentration anywhere; the original impression
  came from the HM3 arm's 5,484 variants, where a spiky-looking Manhattan is the
  null expectation.

In [ ]:
bb2 = bb.sort_values("maf_lo").reset_index(drop=True)
flagged = bb2[bb2["enrich_max"] > ENRICH_FLAG]
clean   = bb2[bb2["enrich_max"] <= ENRICH_FLAG]
print(f"flag = {ENRICH_FLAG}x null\n")

if len(flagged) == 0:
    top = bb2.loc[bb2["enrich_max"].idxmax()]
    print("NO BAND exceeds the flag.")
    print(f"  max enrichment {top['enrich_max']:.2f}x at band {int(top['band'])} "
          f"(MAF {top['maf_lo']:.5f}-{top['maf_hi']:.5f})")
    print("  -> loadings are not measurably concentrated at any frequency in "
          "0.1-1% MAF.")
    print("  -> the original impression most likely came from the HM3 arm, where "
          "5,484 variants\n     make a spiky Manhattan the null expectation.")
elif len(flagged) == len(bb2):
    print("EVERY band exceeds the flag - the effect is present at all "
          "frequencies here,")
    print("so it is not frequency-dependent within 0.1-1% MAF.")
    print(f"  range: {bb2['enrich_max'].min():.2f}x - {bb2['enrich_max'].max():.2f}x")
else:
    fb = [int(x) for x in flagged["band"]]
    cb = [int(x) for x in clean["band"]]
    contiguous = fb == list(range(min(fb), max(fb) + 1))
    print(f"FLAGGED bands: {fb}" + ("" if contiguous else "  (not contiguous)"))
    print(f"CLEAN bands:   {cb}")
    print(f"\naffected frequency range: "
          f"MAF {flagged['maf_lo'].min():.5f}-{flagged['maf_hi'].max():.5f}  "
          f"(MAC {flagged['mac_lo'].min():.0f}-{flagged['mac_hi'].max():.0f})")
    print(f"clean frequency range:    "
          f"MAF {clean['maf_lo'].min():.5f}-{clean['maf_hi'].max():.5f}  "
          f"(MAC {clean['mac_lo'].min():.0f}-{clean['mac_hi'].max():.0f})")

    # where the boundary sits, in MAF and MAC
    if contiguous and min(fb) == int(bb2["band"].min()):
        edge = flagged.iloc[-1]
        print(f"\n-> confined to the LOW-frequency end. Boundary at MAF "
              f"{edge['maf_hi']:.5f} (MAC {edge['mac_hi']:.0f}):")
        print(f"   below it, enrichment up to {flagged['enrich_max'].max():.2f}x; "
              f"above it, at most {clean['enrich_max'].max():.2f}x.")
        print("   Consistent with a low-count artefact - raising the panel MAF "
              "floor above this\n   boundary would remove it.")
    elif contiguous and max(fb) == int(bb2["band"].max()):
        edge = flagged.iloc[0]
        print(f"\n-> confined to the HIGH-frequency end, from MAF "
              f"{edge['maf_lo']:.5f} (MAC {edge['mac_lo']:.0f}) up.")
        print("   Not a low-count artefact; look at what is common enough to "
              "carry an axis here.")
    else:
        print("\n-> flagged bands are scattered, not an onset. Check the "
              "PC x band heatmap:")
        print("   a single PC bright across unrelated bands points at one locus "
              "rather than\n   a frequency effect.")

    worst = flagged.loc[flagged["enrich_max"].idxmax()]
    print(f"\nstrongest: band {int(worst['band'])} at {worst['enrich_max']:.2f}x "
          f"on PC{int(worst['enrich_max_pc'])}")

## Save and upload

In [ ]:
RES_TSV = f"{OUT}/band_metrics.tsv"
PC_TSV  = f"{OUT}/band_enrichment_by_pc.tsv"
res.to_csv(RES_TSV, sep="\t", index=False, float_format="%.6f")
per_pc.to_csv(PC_TSV, sep="\t", index=False, float_format="%.6f")
print(f"wrote {RES_TSV}\nwrote {PC_TSV}")

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{RES_TSV}" "{PC_TSV}" "{OUT_GS}/"
gcloud storage cp "{OUT}/"*.png "{OUT_GS}/figures/"
gcloud storage cp "{OUT}/"*.eigenval "{OUT_GS}/fits/"
"""], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_bands.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_bands.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')